# In this noteboook, we intend to built an ETL pipeline for the company Gans: 
Gans is a startup developing an e-scooter-sharing system. It aspires to operate in the most populous cities all around the world. In each city, the company will have hundreds of e-scooters parked in the streets and allow users to rent them by the minute.

Gans has seen that its operational success depends on something more mundane: having its scooters parked where users need them.

The company wants to anticipate as much as possible scooter movements. Predictive modelling is certainly on the roadmap, but the first step is to collect more data, transform it and store it appropriately.



## Main objective: 
I will develop an ELT pipeline. The pipeline will extract data from 3 different sources, one will be web scrapping, and the other two will be APIs. Then this data will be transformed. The data will be stored in a SQL database, created for that purpose.

In [71]:
import pandas as pd
import sqlalchemy
import pymysql
import requests
from sqlalchemy import create_engine
from getpass import getpass
import os
from dotenv import load_dotenv
from sqlalchemy import text
from bs4 import BeautifulSoup
from datetime import datetime
from lat_lon_parser import parse
from pytz import timezone
from datetime import datetime, timedelta

In [16]:
#Connection to SQL-Database
load_dotenv()

schema = "gans_cities_etl"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(connection_string)

# Method
The following function does:
1. Extract information about one or more german cites: Country, Population, Timestamp_Population, City
2. Stores the data into two SQL tables: cities and populations
3. Creates a unique city_id for each city extracted from wikipedia

In [45]:
#Function to extract cities
def add_city(*cities):
    city_data_list = []

    for city in cities:
        #gettin cities from wikipedia
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {"User-Agent": "Chrome/134.0.0.0"}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, "html.parser")

        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population = int(city_population.replace(",", ""))
        #timestamp
        today = datetime.today().strftime("%Y-%m-%d")
        #forming dataframe
        city_data = pd.DataFrame([{
            "City": city,
            "Country": city_soup.find(class_="infobox-data").get_text(),
            "Population": city_population,
            "Timestamp_Population": today,
            "Latitude": parse(city_soup.find(class_="latitude").get_text()),
            "Longitude": parse(city_soup.find(class_="longitude").get_text())}])
        city_data_list.append(city_data)
        #Exporting to SQL database
        city_exists = pd.read_sql(
            f"SELECT city_id FROM cities WHERE city = '{city}'",
            con=engine)
        if city_exists.empty:
            city_data[["City", "Latitude", "Longitude"]].to_sql(
                "cities",
                con=engine,
                if_exists="append",
                index=False)
        #gettin city_id for further use
        city_id = pd.read_sql(
            f"SELECT city_id FROM cities WHERE city = '{city}'",
            con=engine).iloc[0]["city_id"]
        city_data["city_id"] = city_id
        # Exporting population to SQL database
        city_data[["city_id", "Country", "Population", "Timestamp_Population"]].to_sql(
        "populations",
        con=engine,
        if_exists="append",
        index=False)

    return pd.concat(city_data_list, ignore_index=True)

In [46]:
#City function
add_city("Berlin", "Hamburg", "Cologne", "Munich", "Stuttgart", "Düsseldorf")

,City,Country,Population,Timestamp_Population,Latitude,Longitude,city_id
0,Berlin,Germany,3596999,2026-09-25,52.520000,13.405000,1
1,Hamburg,Germany,1973896,2026-09-25,53.550000,10.000000,2
2,Cologne,Germany,1025523,2026-09-25,50.936389,6.952778,3
3,Munich,Germany,1505036,2026-09-25,48.137500,11.575000,4
4,Stuttgart,Germany,609365,2026-09-25,48.777500,9.180000,5
5,Düsseldorf,Germany,619444,2026-09-25,51.225556,6.776667,6


After extracting our Cities and population Data, we will now build additional functions to get:
1. Weather information for those cities
2. Airport and flights data, as gans wants to target tourists as a main customer group.

## 1. get_weather function() does:
a) get cities data from SQL
b) iterates through all cities
c) uses the coordinates for each city for the api-weather request
d) gets weather data 
e) ensures that a rain column exists, even if there was no rain in the past 3 hours. In this case, a 0 will be inserted
f) selects the columns I am interested in and renames them
g) chnages datetime format

In [ ]:
#Weather-function
API_key = os.getenv("key_weather")

def get_weather(*cities):

    API_key = os.getenv("key_weather")
    #get coordinates, id and cities from SQL
    cities = pd.read_sql(
    "SELECT city_id, city, Latitude, Longitude FROM cities",
    con=engine)
    #Preparing df
    cities_weather_data = pd.DataFrame()
    #determine timezone
    berlin_timezone = timezone("Europe/Berlin")
    # iterating over all cities
    for city_id, city, lat, long in zip(
        cities["city_id"],
        cities["city"],
        cities["Latitude"],
        cities["Longitude"]):
        # API request
        weather_request_cities = requests.get(
            f"https://api.openweathermap.org/data/2.5/forecast?"
            f"lat={lat}&lon={long}&appid={API_key}&units=metric")
        retrieval_time = datetime.now(
            berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
        # Normalize API response
        weather_all_df = pd.json_normalize(
            weather_request_cities.json()["list"])
        # Handle missing rain data
        if "rain.3h" in weather_all_df.columns:
            weather_all_df["rain.3h"] = weather_all_df["rain.3h"].fillna(0)
        else:
            weather_all_df["rain.3h"] = 0
        # Extract weather information
        weather_all_2df = pd.json_normalize(
            weather_all_df["weather"].explode())
        weather_cities = pd.concat([weather_all_df, weather_all_2df[["main", "description"]]], axis=1)
        # Select relevant columns
        weather_cities_final = weather_cities.loc[
            :,
            ["dt_txt",
            "main.temp",
            "main.humidity",
            "main",
            "description",
            "rain.3h",
            "pop",
            "wind.speed"]]
        # Rename columns
        weather_cities_final = weather_cities_final.rename(
            columns={
                "dt_txt": "forecast_time",
                "main.temp": "temperature",
                "main.humidity": "humidity",
                "main": "forecast",
                "description": "weather_info",
                "rain.3h": "rain_in_last_3h",
                "pop": "rain_propability",
                "wind.speed": "wind_speed"})
        #Integrating id, city, and retrieval date into df
        weather_cities_final["data_retrieved_at"] = retrieval_time
        weather_cities_final["City"] = city
        weather_cities_final["city_id"] = city_id
        # building df based on loop results
        cities_weather_data = pd.concat(
            [cities_weather_data, weather_cities_final],
            ignore_index=True)
    # datetime formating
    cities_weather_data["forecast_time"] = pd.to_datetime(
        cities_weather_data["forecast_time"])
    cities_weather_data["data_retrieved_at"] = pd.to_datetime(
        cities_weather_data["data_retrieved_at"])

    return cities_weather_data

In [ ]:
# This function allways insert weather information for all cities into SQL 
# So far, it does not work when writing a single city into the brackets!
weather_df = get_weather()

## The following function extracts airports for our cities

In [68]:

# Function to extract and insert airports
def add_airports(*cities):
    API_key = os.getenv("key_flights")
    # Get cities and coordinates from SQL
    cities_df = pd.read_sql(
        "SELECT city_id, city, Latitude, Longitude FROM cities",
        con=engine)
    all_airports = []
    # API headers
    headers = {"x-rapidapi-key": API_key,
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type": "application/json"}
    for city in cities:
        city_data = cities_df[cities_df["city"] == city].iloc[0]
        city_id = city_data["city_id"]
        lat = city_data["Latitude"]
        lon = city_data["Longitude"]
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat": lat, "lon": lon, "radiusKm": "50", "limit": "10", "withFlightInfoOnly": "true"}
        # Make API request
        response = requests.get(
            url,
            headers=headers,
            params=querystring)
        print(city, response.status_code, response.json())
        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(
                data.get("items", []))
            airports["city_id"] = city_id
            all_airports.append(airports)
    all_airports_final = pd.concat(
        all_airports,
        ignore_index=True)
    #renaming columns
    all_airports_final = all_airports_final.rename(
        columns={"icao" : "airport_icao", "municipalityName": "City", "location.lat": "location_lat",
        "location.lon": "location_lon"})
    # selecting columns to prevent localCode error
    all_airports_final = all_airports_final[
    ["airport_icao", "iata", "name", "shortName", "City",
    "countryCode", "timeZone", "location_lat", "location_lon", "city_id"]]
    all_airports_final.to_sql(
        "airports",
        con=engine,
        if_exists="append",
        index=False)

    return all_airports_final


In [70]:
# Function : It adds airport data to SQL for every city typed into the function
add_airports("Stuttgart", "Munich")

Stuttgart 200 {'searchBy': {'lat': 48.7775, 'lon': 9.18}, 'count': 1, 'items': [{'icao': 'EDDS', 'iata': 'STR', 'name': 'Stuttgart', 'shortName': 'Stuttgart', 'municipalityName': 'Stuttgart', 'location': {'lat': 48.6899, 'lon': 9.22196}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
Munich 200 {'searchBy': {'lat': 48.1375, 'lon': 11.575}, 'count': 1, 'items': [{'icao': 'EDDM', 'iata': 'MUC', 'name': 'Munich', 'shortName': 'Munich', 'municipalityName': 'Munich', 'location': {'lat': 48.3538, 'lon': 11.7861}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}


,airport_icao,iata,name,shortName,City,countryCode,timeZone,location_lat,location_lon,city_id
0,EDDS,STR,Stuttgart,Stuttgart,Stuttgart,DE,Europe/Berlin,48.6899,9.22196,5
1,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.3538,11.78610,4


## The following function will insert flight data into our database. Upcoming flights (only arrivals) to the airports of our airports will be inserted into SQL

In [ ]:
# Function to extract and insert flights
def add_flights(*cities):
    API_key = os.getenv("key_flights")

    # Get airports for selected cities from SQL
    airports_df = pd.read_sql(
        f"""SELECT a.airport_icao
        FROM airports a JOIN cities c ON a.city_id = c.city_id
        WHERE c.city IN ({",".join([f"'{city}'" for city in cities])})
        """,con=engine)
    # Define 12-hour time period
    start_time = datetime.now()
    end_time = start_time + timedelta(hours=12)
    start_time = start_time.strftime("%Y-%m-%dT%H:%M")
    end_time = end_time.strftime("%Y-%m-%dT%H:%M")
    all_flights = []
    # API headers
    headers = {"x-rapidapi-key": API_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"}

    for ica in airports_df["airport_icao"]:
        url = (
            f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/"
            f"{ica}/{start_time}/{end_time}")
        #refining our request:
        querystring = {"withLeg": "true", "direction": "Arrival", "withCancelled": "false",
            "withCodeshared": "true", "withCargo": "false", "withPrivate": "true",
            "withLocation": "true"}
        response_flights = requests.get(url, headers=headers, params=querystring)
        #Orders to execute when request was successful(200)
        if response_flights.status_code == 200:
            data = response_flights.json()
            flights = pd.json_normalize(data["arrivals"])
            flights["arrival.scheduledTime.local"] = (
                flights["arrival.scheduledTime.local"]
                #str.replace changes: 2026-09-25 13:30+02:00 to 2026-09-25 13:30
                .str.replace(r"\+02:00$", "", regex=True))
            flights["arrival.revisedTime.local"] = (
                flights["arrival.revisedTime.local"]
                .str.replace(r"\+02:00$", "", regex=True))
            flights["icao"] = ica
            all_flights.append(flights)
    all_flights_final = pd.concat(all_flights, ignore_index=True)
    # Selecting relevant columns
    all_flights_final = all_flights_final[["number", "status",
            "departure.airport.icao", "departure.airport.countryCode",
            "icao", "arrival.scheduledTime.local", "arrival.revisedTime.local"]]
    # Renaming columns
    all_flights_final = all_flights_final.rename(columns={
            "departure.airport.icao": "departure_airport_icao",
            "departure.airport.countryCode": "departure_airport_countryCode",
            "icao": "airport_icao",
            "arrival.scheduledTime.local": "arrival_scheduled",
            "arrival.revisedTime.local": "arrival_revised"})
    # Export to SQL
    all_flights_final.to_sql(
        "flights",
        con=engine,
        if_exists="append",
        index=False)

    return all_flights_final

In [73]:
add_flights("Berlin", "Hamburg", "Cologne", "Munich", "Stuttgart")

,number,status,departure_airport_icao,departure_airport_countryCode,airport_icao,arrival_scheduled,arrival_revised
0,BA 990,Approaching,EGLL,gb,EDDB,2026-09-25 13:25,2026-09-25 13:32
1,DY 1104,Expected,ENGM,no,EDDB,2026-09-25 13:30,2026-09-25 13:35
2,XQ 946,Expected,LTBJ,tr,EDDB,2026-09-25 13:40,2026-09-25 13:46
3,SK 1677,Expected,EKCH,dk,EDDB,2026-09-25 13:55,2026-09-25 13:51
4,MF 9333,Delayed,EHAM,nl,EDDB,2026-09-25 13:35,2026-09-25 14:04
...,...,...,...,...,...,...,...
1082,EW 2231,Expected,LGKO,gr,EDDS,2026-09-25 23:05,2026-09-25 23:05
1083,EW 2587,Expected,LEPA,es,EDDS,2026-09-25 23:05,2026-09-25 23:05
1084,EW 2823,Expected,LIMC,it,EDDS,2026-09-25 23:15,2026-09-25 23:15
1085,EW 2683,Expected,LGTS,gr,EDDS,2026-09-25 23:25,2026-09-25 23:25
